SEFIN DEUDA PUBLICA (EXCEL) TRIMESTRAL

In [11]:
import pandas as pd
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)



url="C:\\Users\\HN11133\\Desktop\\Tareas\\API\\indicadores_deuda_1trimestr2025.xlsx"


tabla=pd.read_excel(url)

df_largo=tabla

df_largo["VARIACION"]=df_largo.groupby('NOMBRE_INDICADOR')['VALOR'].pct_change().round(4)
df_largo["DESCRIPCION"]="SEFIN"
df_largo["PERIODICIDAD"]="Trimestral"
df_largo["TIPO"]="Deuda Publica"


df_largo.head(10)

,NOMBRE_INDICADOR,FECHA,VALOR,VARIACION,DESCRIPCION,PERIODICIDAD,TIPO
0,Deuda Interna,2015-03-30,3235.485223,NaN,SEFIN,Trimestral,Deuda Publica
1,Deuda Interna,2015-06-30,3280.369660,0.0139,SEFIN,Trimestral,Deuda Publica
2,Deuda Interna,2015-09-30,3338.498771,0.0177,SEFIN,Trimestral,Deuda Publica
3,Deuda Interna,2015-12-30,3461.857755,0.0370,SEFIN,Trimestral,Deuda Publica
4,Deuda Interna,2016-03-30,3470.104282,0.0024,SEFIN,Trimestral,Deuda Publica
5,Deuda Interna,2016-06-30,3576.449613,0.0306,SEFIN,Trimestral,Deuda Publica
6,Deuda Interna,2016-09-30,3510.813853,-0.0184,SEFIN,Trimestral,Deuda Publica
7,Deuda Interna,2016-12-30,3929.384992,0.1192,SEFIN,Trimestral,Deuda Publica
8,Deuda Interna,2017-03-30,3790.376970,-0.0354,SEFIN,Trimestral,Deuda Publica
9,Deuda Interna,2017-06-30,3919.100000,0.0340,SEFIN,Trimestral,Deuda Publica


In [12]:
################           INSERT TABLE / DATA           #################   #      
from azure.identity import InteractiveBrowserCredential
import pandas as pd
from Server import AZURE
from tqdm import tqdm
from sqlalchemy import create_engine, text

credential = InteractiveBrowserCredential()

server = AZURE
database = 'sqlpooldwhandr01'
schema = 'HN_NAP_HO_MISRIESGOS_F'
tabla = 'INDICADORES_MACROECONOMICOS'
driver = "ODBC Driver 17 for SQL Server"

connection_string = (
        f"DRIVER={driver};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Encrypt=yes;"
        f"TrustServerCertificate=no;"
        f"Authentication=ActiveDirectoryInteractive;"
    )

connection_uri = f"mssql+pyodbc:///?odbc_connect={connection_string}"
engine = create_engine(connection_uri, fast_executemany=True)


query = text("""
SELECT COLUMN_NAME, DATA_TYPE , CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_NAME = :tablita
AND TABLE_SCHEMA = :esquema
""")

with engine.connect() as conn:
    result = conn.execute(query, {"tablita": tabla, "esquema":schema})
    columns_types = {row[0]: [row[1] , row[2]] for row in result}

In [13]:
#-----------------------------############### INSERT ##################------------------------------------#

#df_resultado['DATE_TIME']=pd.to_datetime(df_resultado['DATE_TIME']).dt.strftime('%Y-%m-%d')
data_frame=df_largo
chunksize = 100

for start in tqdm(range(0, len(data_frame), chunksize), desc="Insertando datos"):
    end = min(start + chunksize, len(data_frame))
    chunk = data_frame.iloc[start:end]

    try:
        chunk.to_sql(tabla, 
                     con=engine, 
                     schema=schema, 
                     if_exists='append', 
                     index=False, 
                     chunksize=chunksize)
    except Exception as e:
        print(f"Error al insertar datos en la base de datos: {e}")
    #break
print('Proceso de insercion completado')

Insertando datos: 100%|██████████| 2/2 [00:11<00:00,  5.97s/it]

Proceso de insercion completado
